# 📒 Bài 2 — Training loop & MLP

**Tuần 12** · ⭐ **Notebook quan trọng nhất Phase 05**

> Chạy trước: `python curriculum/05-deep-learning/tao_du_lieu.py`

Bạn sắp viết vòng lặp mà **mọi** model AI — từ MLP nhỏ đến GPT — đều dùng.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

MAU = ["#4269d0", "#efb118", "#3ca951", "#ff725c", "#a463f2"]
MAU_CHINH, MAU_NHAN = MAU[0], MAU[3]

torch.manual_seed(42)

DATA = Path("../../../data")
if not (DATA / "hinh_khoi.npz").exists():
    DATA = Path("data")

d = np.load(DATA / "hinh_khoi.npz")
TEN_LOP = list(d["ten_lop"])

X_train = torch.tensor(d["X_train"])          # (8000, 28, 28)
y_train = torch.tensor(d["y_train"])
X_test = torch.tensor(d["X_test"])
y_test = torch.tensor(d["y_test"])

print(f"train {tuple(X_train.shape)}   test {tuple(X_test.shape)}")
print("lớp:", TEN_LOP)

## 1. Nhìn dữ liệu trước (thói quen từ Phase 03)

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(12, 4.2))
for ax, i in zip(axes.flat, range(12)):
    ax.imshow(X_train[i], cmap="gray", vmin=0, vmax=1)
    ax.set_title(TEN_LOP[y_train[i]], fontsize=9)
    ax.axis("off")
plt.suptitle("Ảnh có nhiễu, xoay, dịch chuyển, đổi kích thước", y=1.02)
plt.tight_layout(); plt.show()

print("Phân bố lớp:", torch.bincount(y_train).tolist())

> Chú ý: hình **xoay** ngẫu nhiên và nằm ở **vị trí** khác nhau. Đây chính là thứ làm khó MLP — và là chỗ CNN toả sáng ở bài 3.

## 2. Chuẩn bị dữ liệu cho MLP

In [ ]:
Xtr = X_train.flatten(1)          # (8000, 28, 28) -> (8000, 784)
Xte = X_test.flatten(1)

print("Sau flatten:", tuple(Xtr.shape))
print()
print("→ MLP nhìn ảnh như một dãy 784 số rời rạc.")
print("  Thông tin 'pixel nào cạnh pixel nào' bị VỨT BỎ hoàn toàn.")

## 3. Xây MLP

In [ ]:
def tao_mlp(vao, an, ra):
    lop = []
    truoc = vao
    for k in an:
        lop += [nn.Linear(truoc, k), nn.ReLU()]
        truoc = k
    lop.append(nn.Linear(truoc, ra))      # lớp cuối: KHÔNG ReLU, KHÔNG Softmax
    return nn.Sequential(*lop)


torch.manual_seed(0)
model = tao_mlp(784, [128, 64], 4)
print(model)
print(f"\nTổng tham số: {sum(p.numel() for p in model.parameters()):,}")

> ⚠️ **Lớp cuối trả về LOGITS thô — không ReLU, không Softmax.**
>
> `nn.CrossEntropyLoss` **đã bao gồm** softmax bên trong. Thêm `nn.Softmax()` vào model = áp dụng hai lần → model học rất kém mà **không báo lỗi gì**. Đây là bẫy số 1 của người mới.

## 4. ⭐⭐ Training loop — bốn bước

In [ ]:
def tao_batch(X, y, batch_size, seed=0):
    g = torch.Generator().manual_seed(seed)
    idx = torch.randperm(len(X), generator=g)
    return [(X[idx[i:i+batch_size]], y[idx[i:i+batch_size]])
            for i in range(0, len(X), batch_size)]


def train_mot_epoch(model, batches, ham_loss, opt):
    model.train()
    tong = 0.0
    for xb, yb in batches:
        y_pred = model(xb)                 # ① FORWARD
        loss = ham_loss(y_pred, yb)        # ② LOSS

        opt.zero_grad()                    # ③ XOÁ gradient cũ
        loss.backward()                    #    BACKWARD
        opt.step()                         # ④ UPDATE

        tong += loss.item()                # .item() — đừng quên!
    return tong / len(batches)


@torch.no_grad()
def danh_gia(model, X, y, ham_loss):
    model.eval()
    logits = model(X)
    return {
        "loss": ham_loss(logits, y).item(),
        "acc": (logits.argmax(1) == y).float().mean().item(),
    }

**Sáu dòng trong `train_mot_epoch` là toàn bộ deep learning.** Đối chiếu với Phase 02:

| Phase 02 (bạn tự viết bằng NumPy) | PyTorch |
|---|---|
| `y_pred = X @ w + b` | `y_pred = model(xb)` |
| `loss = np.mean((y_pred-y)**2)` | `loss = ham_loss(y_pred, yb)` |
| `grad_w = (2/n) * X.T @ sai_so` | `loss.backward()` |
| `w = w - lr * grad_w` | `opt.step()` |

PyTorch chỉ thay bạn làm **bước 3** — bước khó nhất. Ba bước kia bạn vẫn tự viết.

**Hai chi tiết dễ bỏ sót:**
- `.item()` khi cộng dồn loss — nếu không, bạn giữ lại đồ thị tính toán của **mọi** batch → rò rỉ bộ nhớ
- `model.train()` / `model.eval()` — bắt buộc khi model có Dropout hoặc BatchNorm

## 5. Train thật

In [ ]:
import time

torch.manual_seed(0)
model = tao_mlp(784, [128, 64], 4)
ham_loss = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

lich_su = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
t0 = time.perf_counter()

for epoch in range(15):
    train_mot_epoch(model, tao_batch(Xtr, y_train, 64, seed=epoch), ham_loss, opt)

    tr = danh_gia(model, Xtr, y_train, ham_loss)
    va = danh_gia(model, Xte, y_test, ham_loss)
    for k, v in [("train_loss", tr["loss"]), ("train_acc", tr["acc"]),
                 ("val_loss", va["loss"]), ("val_acc", va["acc"])]:
        lich_su[k].append(v)

    if epoch % 3 == 0 or epoch == 14:
        print(f"  epoch {epoch+1:>2}  train_loss {tr['loss']:.4f}  "
              f"val_loss {va['loss']:.4f}  val_acc {va['acc']:.4f}")

print(f"\n  Xong trong {time.perf_counter()-t0:.1f} giây (CPU)")
print(f"  Độ chính xác cuối: {lich_su['val_acc'][-1]:.1%}")

## 6. ⭐ Đường cong học — công cụ chẩn đoán mạnh nhất

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2)

a1.plot(lich_su["train_loss"], lw=2, color=MAU[0], label="train")
a1.plot(lich_su["val_loss"], lw=2, color=MAU[3], label="validation")
a1.set_xlabel("epoch"); a1.set_ylabel("loss"); a1.set_title("Loss")
a1.legend(frameon=False)

a2.plot(lich_su["train_acc"], lw=2, color=MAU[0], label="train")
a2.plot(lich_su["val_acc"], lw=2, color=MAU[3], label="validation")
a2.set_xlabel("epoch"); a2.set_ylabel("accuracy"); a2.set_title("Độ chính xác")
a2.legend(frameon=False)

plt.tight_layout(); plt.show()

print(f"  Khoảng cách train-val: {lich_su['train_acc'][-1] - lich_su['val_acc'][-1]:+.4f}")

```
  Loss                              Loss
   │╲                                │╲
   │ ╲___ train                      │ ╲___ train
   │     ╲_____                      │     ╲______
   │  ╲___                           │  ╲___    ___╱─── val  ← TĂNG TRỞ LẠI
   │      ╲____ val                  │      ╲__╱
   └──────────────▶ epoch            └──────────────▶ epoch
        ✅ Tốt                        ⚠️ Overfit từ đây
```

| Dấu hiệu | Nghĩa | Chữa |
|---|---|---|
| val_loss **tăng trở lại** | Overfit | Early stopping, dropout, augmentation, thêm dữ liệu |
| Cả hai **không giảm** | Chưa học được | Tăng lr, model lớn hơn, **kiểm tra lại dữ liệu** |
| Loss ra `nan` | lr quá lớn | Giảm lr 10 lần |
| train ≪ val | Overfit | Như trên |

> 🔑 **Luôn vẽ đường cong của cả train và validation.** Nó cho biết nhiều hơn bất kỳ con số đơn lẻ nào.

## 7. Learning rate — vẫn là siêu tham số quan trọng nhất

In [ ]:
def thu_lr(lr, so_epoch=6):
    torch.manual_seed(0)
    m = tao_mlp(784, [128, 64], 4)
    o = torch.optim.Adam(m.parameters(), lr=lr)
    ls = []
    for e in range(so_epoch):
        train_mot_epoch(m, tao_batch(Xtr, y_train, 64, seed=e), ham_loss, o)
        ls.append(danh_gia(m, Xte, y_test, ham_loss)["loss"])
    return ls


for lr in [1e-5, 1e-3, 1e-1]:
    ls = thu_lr(lr)
    nhan = f"{ls[-1]:.4f}" if np.isfinite(ls[-1]) else "nan"
    plt.plot(ls, marker="o", lw=2, label=f"lr={lr:g} (val_loss cuối: {nhan})")

plt.xlabel("epoch"); plt.ylabel("val loss"); plt.ylim(0, 2)
plt.legend(frameon=False); plt.title("Ảnh hưởng của learning rate")
plt.show()

> 💡 `1e-3` với Adam là điểm khởi đầu tốt cho gần như mọi bài toán. Quá nhỏ → học chậm; quá lớn → dao động hoặc `nan`.

## 8. Model sai ở đâu?

In [ ]:
with torch.no_grad():
    model.eval()
    logits = model(Xte)
    du_doan = logits.argmax(1)

cm = torch.zeros(4, 4, dtype=torch.int)
for t, p in zip(y_test, du_doan):
    cm[t, p] += 1

fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(4), TEN_LOP, rotation=30, ha="right")
ax.set_yticks(range(4), TEN_LOP)
ax.set_xlabel("dự đoán"); ax.set_ylabel("thật")
ax.grid(visible=False)
for i in range(4):
    for j in range(4):
        ax.text(j, i, int(cm[i, j]), ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "#222")
plt.title("Confusion matrix — MLP")
plt.tight_layout(); plt.show()

for i, ten in enumerate(TEN_LOP):
    print(f"  {ten:<10} đúng {cm[i, i]:>4} / {cm[i].sum():>4}  = {cm[i,i]/cm[i].sum():.1%}")

In [ ]:
# Nhìn những ảnh model đoán SAI
sai = (du_doan != y_test).nonzero().flatten()[:12]

fig, axes = plt.subplots(2, 6, figsize=(12, 4.2))
for ax, i in zip(axes.flat, sai):
    ax.imshow(X_test[i], cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"thật: {TEN_LOP[y_test[i]]}\nđoán: {TEN_LOP[du_doan[i]]}", fontsize=8)
    ax.axis("off")
plt.suptitle(f"{len(sai)} trong số {(du_doan != y_test).sum()} ảnh bị đoán sai", y=1.03)
plt.tight_layout(); plt.show()

> 🔍 **Luôn nhìn những mẫu model đoán sai.** Nó cho biết model đang nhầm *kiểu gì* — thông tin mà accuracy không bao giờ cho bạn. Ở đây bạn sẽ thấy MLP nhầm nhiều ở các hình bị **xoay** — đúng như dự đoán.

## 9. Tổng kết

```python
# Khung train dùng lại cho mọi bài toán
ham_loss = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(so_epoch):
    model.train()
    for xb, yb in batches:
        loss = ham_loss(model(xb), yb)     # ① forward + ② loss
        opt.zero_grad()                    # ③ xoá gradient cũ
        loss.backward()                    #    backward
        opt.step()                         # ④ update

    model.eval()
    with torch.no_grad():
        ...                                # đánh giá, ghi lịch sử
```

**Kết quả MLP: ~82%.** Bài 3 sẽ dùng CNN trên **cùng dữ liệu** và đạt ~96%. Khác biệt 14 điểm đó đến từ đâu — đó là nội dung bài tiếp theo.

## ✍️ Bài tập

```bash
pytest tests/phase05/test_ex02.py -v
```